# Phase 2: Panel Feature Engineering
research_v2.db — coverage diagnostics first, then feature table build.

In [ ]:
import os
import sys
os.chdir("../..")

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from config import DATA_ROOT, NSE_DB_PATH

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 160)

DB_PATH = "research_v2.db"  # point at the real candL warehouse path or attach it below
con = duckdb.connect(DB_PATH)
con.execute("SHOW TABLES").df()

,name


In [ ]:
RESEARCH_DB_PATH = DATA_ROOT / "research.db"
con = duckdb.connect(RESEARCH_DB_PATH)
con.execute(f"ATTACH '{NSE_DB_PATH}' AS nse (READ_ONLY)") 

## Options coverage per ticker
Since liquidity concentrates in the front month, this checks per (ticker, expiry-bucket) rather than flat per-ticker, so the ragged near/far-month pattern is visible instead of averaged away.

In [ ]:
coverage_sql = """
WITH strike_counts AS (
    SELECT
        trade_date,
        ticker,
        expiry,
        COUNT(DISTINCT instrument_key) AS n_strikes,
        SUM(open_interest) AS total_oi
    FROM market_data_daily m
    JOIN instruments i USING (instrument_key)
    WHERE i.instrument_type = 'STO'
    GROUP BY 1,2,3
),
labeled AS (
    SELECT *,
        DATE_DIFF('day', trade_date, expiry) AS dte,
        CASE
            WHEN DATE_DIFF('day', trade_date, expiry) <= 35 THEN 'near'
            WHEN DATE_DIFF('day', trade_date, expiry) <= 65 THEN 'mid'
            ELSE 'far'
        END AS month_bucket
    FROM strike_counts
),
usable AS (
    SELECT *, (n_strikes >= 6 AND total_oi >= 1000) AS is_usable
    FROM labeled
)
SELECT
    ticker,
    month_bucket,
    COUNT(*) AS day_count,
    AVG(n_strikes) AS avg_strikes,
    AVG(total_oi) AS avg_oi,
    AVG(CAST(is_usable AS INT)) AS pct_usable
FROM usable
GROUP BY 1,2
ORDER BY ticker, month_bucket
"""
coverage = con.execute(coverage_sql).df()
coverage.head(20)

## F&O eligibility window per ticker
First/last date a ticker has *any* STF/STO instrument — flags entries/exits from the F&O list so the panel join doesn't silently zero-fill stocks that weren't F&O-eligible yet.

In [ ]:
eligibility_sql = """
SELECT
    i.ticker,
    MIN(m.trade_date) AS first_fo_date,
    MAX(m.trade_date) AS last_fo_date,
    COUNT(DISTINCT m.trade_date) AS n_days_with_fo
FROM market_data_daily m
JOIN instruments i USING (instrument_key)
WHERE i.instrument_type IN ('STF','STO')
GROUP BY 1
ORDER BY first_fo_date DESC
"""
eligibility = con.execute(eligibility_sql).df()
eligibility.head(20)

## Near-month-only usable coverage summary
Single number per ticker: % of trading days within its own F&O-eligible window where the near-month options chain is usable. This is the figure that decides full-panel vs tiered-feature handling.

In [ ]:
near_only = coverage[coverage.month_bucket == 'near']
summary = near_only.merge(eligibility, on='ticker', how='left')
summary['coverage_ratio'] = summary['day_count'] / summary['n_days_with_fo']
summary = summary[['ticker','n_days_with_fo','day_count','pct_usable','coverage_ratio']]
summary = summary.sort_values('coverage_ratio')
summary

In [ ]:
# Quick triage: how many tickers fall into each reliability tier
bins = [0, 0.5, 0.8, 0.95, 1.01]
labels = ['unreliable (<50%)', 'patchy (50-80%)', 'mostly_good (80-95%)', 'reliable (95-100%)']
summary['tier'] = pd.cut(summary['coverage_ratio'], bins=bins, labels=labels)
summary['tier'].value_counts().sort_index()